# 04 - Allocation pressure review

This notebook adds one extra flag to the final allocation. The flag marks MSOAs that have unusually low allocated FTE compared with their forecast weighted demand.

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in ["notebooks", "allocation"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"

MIN_WEIGHTED_DEMAND = 0.0
PRESSURE_Z_THRESHOLD = -1.5


with sqlite3.connect(ALLOC_DB_PATH) as conn:
    final_allocation = pd.read_sql_query("SELECT * FROM final_allocation;", conn)

# SQLite reads boolean columns as 0/1, so convert back before saving CSV outputs.
if "uncertainty_flag" in final_allocation.columns:
    final_allocation["uncertainty_flag"] = final_allocation["uncertainty_flag"].astype(bool)

# Remove the flag if this notebook is rerun.
final_allocation = final_allocation.drop(
    columns=[c for c in ["allocation_pressure_flag"] if c in final_allocation.columns]
)

print("rows:", len(final_allocation))
print("months:", sorted(final_allocation["month"].unique()))

## Calculate MSOA-level resource intensity

The allocation table has one row per MSOA, month, and bucket. For this check we first add the buckets together so each MSOA/month has one total allocation and one total weighted demand.

In [ ]:
pressure_review = (
    final_allocation
    .groupby(["pfa_code", "pfa_name", "msoa_code", "msoa_name", "month"], as_index=False)
    .agg(
        weighted_demand=("weighted_demand", "sum"),
        total_alloc=("total_alloc", "sum"),
        total_capacity=("total_capacity", "max"),
        uncertainty_flag=("uncertainty_flag", "max"),
        variable_alloc=("variable_alloc", "sum"),
    )
)

# If a force has no demand-led allocation, do not use it for the pressure flag.
# This keeps Great Manchester out because its allocation is baseline-only.
force_reliability = (
    pressure_review
    .groupby("pfa_code", as_index=False)
    .agg(total_variable_alloc=("variable_alloc", "sum"))
)
force_reliability["demand_reliability"] = np.where(
    force_reliability["total_variable_alloc"] > 0,
    1.0,
    0.0,
)
pressure_review = pressure_review.merge(
    force_reliability[["pfa_code", "demand_reliability"]],
    on="pfa_code",
    how="left",
)

pressure_review["fte_per_100_weighted_demand"] = np.where(
    pressure_review["weighted_demand"] > 0,
    pressure_review["total_alloc"] / pressure_review["weighted_demand"] * 100,
    np.nan,
)

pressure_review["eligible_for_pressure_flag"] = (
    (pressure_review["weighted_demand"] > MIN_WEIGHTED_DEMAND)
    & (pressure_review["demand_reliability"] > 0)
    & pressure_review["fte_per_100_weighted_demand"].notna()
    & (pressure_review["fte_per_100_weighted_demand"] > 0)
)

pressure_review.head()

## Flag low allocation pressure outliers

The flag uses a z-score within each police force and month. We use the log of resource intensity because the raw ratio is skewed.

`allocation_pressure_flag = True` when the MSOA is at least `1.5` standard deviations below the force/month average.

In [ ]:
pressure_review["log_resource_intensity"] = np.log(pressure_review["fte_per_100_weighted_demand"])

force_month_stats = (
    pressure_review[pressure_review["eligible_for_pressure_flag"]]
    .groupby(["pfa_code", "month"])["log_resource_intensity"]
    .agg(force_month_log_mean="mean", force_month_log_sd="std")
    .reset_index()
)

pressure_review = pressure_review.merge(force_month_stats, on=["pfa_code", "month"], how="left")
pressure_review["log_pressure_z_score"] = (
    (pressure_review["log_resource_intensity"] - pressure_review["force_month_log_mean"])
    / pressure_review["force_month_log_sd"]
)

pressure_review["allocation_pressure_flag"] = (
    pressure_review["eligible_for_pressure_flag"]
    & (pressure_review["log_pressure_z_score"] <= PRESSURE_Z_THRESHOLD)
)

pressure_review = pressure_review[
    [
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "month",
        "weighted_demand",
        "total_alloc",
        "total_capacity",
        "fte_per_100_weighted_demand",
        "log_pressure_z_score",
        "allocation_pressure_flag",
        "uncertainty_flag",
    ]
]

pressure_review[pressure_review["allocation_pressure_flag"]].head()

## Copy flag back to final allocation

The flag is calculated once per MSOA/month, then copied onto each bucket row for that MSOA/month.

In [ ]:
pressure_flags = pressure_review[["pfa_code", "msoa_code", "month", "allocation_pressure_flag"]]

final_allocation = final_allocation.merge(
    pressure_flags,
    on=["pfa_code", "msoa_code", "month"],
    how="left",
)
final_allocation["allocation_pressure_flag"] = (
    final_allocation["allocation_pressure_flag"].fillna(False).astype(bool)
)

print("flagged MSOA-months:", int(pressure_review["allocation_pressure_flag"].sum()))
print("flagged final allocation rows:", int(final_allocation["allocation_pressure_flag"].sum()))

## Save

In [ ]:
final_allocation_path = DATA_DIR / "final_allocation.csv"

final_allocation.to_csv(final_allocation_path, index=False)

with sqlite3.connect(ALLOC_DB_PATH) as conn:
    final_allocation.to_sql("final_allocation", conn, if_exists="replace", index=False)

print("updated final_allocation")